# 🌏 模組活動一｜公開資料、API 與 Prompt-driven Notebook
### 用 Prompt 讓 AI 寫程式、由你看懂與驗證：真實抓取花蓮站空品資料（現況＋歷史）
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rlirdo/envchem-ai-module-kit/blob/main/notebooks/M1_moenv_api_fetch.ipynb)　｜　[懶人包網站](https://rlirdo.github.io/envchem-ai-module-kit/)　｜　[本模組操作手冊](https://rlirdo.github.io/envchem-ai-module-kit/m1.html)　｜　[Prompt 指南](https://rlirdo.github.io/envchem-ai-module-kit/prompts.html)

**今天的任務**：你是花蓮縣環保局的「數據科技特派員」。請用 Prompt 讓 AI 寫出串接**環境部 Open API** 的程式，抓到**花蓮站**的現況與近一個月歷史空品資料，存成 CSV，下週做清洗與畫圖。

| 步驟 | 你要做的事 | 成功的樣子 |
|---|---|---|
| 0 | 設定金鑰（🔑 密鑰） | 看到「已讀到金鑰」 |
| 1 | 組出 API 網址 | 印出遮住金鑰的網址 |
| 2 | 第一次和 AI 對話：生成程式碼 | 看到全臺測站表格 |
| 3 | 把錯誤丟回 AI 除錯 | 錯誤消失 |
| 4 | 篩出花蓮（現況） | 只剩 1 列花蓮 |
| 5 | 抓花蓮近一個月歷史資料 | 約 700 列逐時資料 |
| 6 | 匯出 CSV、下載 | 電腦裡有 CSV 檔 |

> 📌 規則：先自己寫 Prompt 給 AI；**卡關超過 5 分鐘**再展開「參考解答」。所有你給 AI 的提示詞，請貼到學習單「Prompts 紀錄」。

<sub>NDHU 自資系 游豐兆 博士生製作｜指導教授 楊悠娟 副教授｜國立東華大學 115-1 環境化學 × 生成式 AI 融入專業課程計畫</sub>

## 步驟 0｜設定 API 金鑰（金鑰＝你的帳號密碼，不能寫進程式碼！）
1. 點左側工具列的 **🔑（密鑰）** → **新增密鑰**
2. 名稱填 `MOENV_API_KEY`，值貼上環境部寄給你的金鑰
3. 打開「**筆記本存取權**」開關 → 執行下面這格

In [1]:
# 讀取 API 金鑰：請先在左側「🔑 密鑰」新增 MOENV_API_KEY，並打開「筆記本存取權」
try:
    from google.colab import userdata
    API_KEY = userdata.get('MOENV_API_KEY')
except Exception:                      # 不在 Colab（或還沒設定密鑰）時，改用環境變數或手動輸入
    import os, getpass
    API_KEY = os.environ.get('MOENV_API_KEY') or getpass.getpass('貼上你的 API Key（不會顯示）：')
print('✅ 已讀到金鑰，長度', len(API_KEY), '（金鑰本身不印出來）')

✅ 已讀到金鑰，長度 36 （金鑰本身不印出來）


In [2]:
# 載入懶人包的「環境部 API 常用代碼庫」（只做一次）
import os, urllib.request
if not os.path.exists('moenv_api.py'):
    urllib.request.urlretrieve('https://rlirdo.github.io/envchem-ai-module-kit/moenv_api.py', 'moenv_api.py')
import moenv_api as ma
import pandas as pd
print('✅ 代碼庫已載入：', list(ma.DATASETS))

✅ 代碼庫已載入： ['aqx_p_432', 'aqx_p_488', 'aqx_p_13', 'aqx_p_07']


## 步驟 1｜認識 API 網址：把一長串拆成積木

In [3]:
# API 網址的積木：基底 + DataID + 參數（api_key、limit、format…）
BASE = 'https://data.moenv.gov.tw/api/v2/'
DATA_ID = 'aqx_p_432'          # 空氣品質指標(AQI)－現況，每小時更新
url = f'{BASE}{DATA_ID}?api_key={API_KEY}&limit=1000&format=JSON'
print(url.replace(API_KEY, '********'))   # 印出來時把金鑰遮起來

https://data.moenv.gov.tw/api/v2/aqx_p_432?api_key=********&limit=1000&format=JSON


### 看資料長什麼樣子（等一下貼 1–2 筆給 AI 當樣本）
環境部的欄位名稱是**英文**（`sitename`、`publishtime`、`pm2.5`、`o3`、`no2`）。把樣本貼給 AI，它就不用猜「監測時間」叫什麼。

In [4]:
import json
sample = ma.fetch('aqx_p_432', API_KEY, limit=2, filters='sitename,EQ,花蓮')   # 花蓮站現況通常只有 1 筆
keep = ['sitename', 'county', 'aqi', 'pm2.5', 'o3', 'no2', 'publishtime']
for row in sample[keep].to_dict('records'):
    print(json.dumps(row, ensure_ascii=False))   # 複製這幾行，貼進改良版 Prompt 的②

{"sitename": "花蓮", "county": "花蓮縣", "aqi": "23", "pm2.5": "0", "o3": "30", "no2": "0", "publishtime": "2026/10/18 15:00:00"}


## 步驟 2｜第一次和 AI 對話：生成程式碼
點 Colab 上方「**＋程式碼**」新增一格，再點格子裡的 **「生成」**（或右上角 **Gemini** 圖示），貼上下面的提示詞（**把網址換成你自己的**，金鑰用 `API_KEY` 變數代替，不要貼真的金鑰）：

```
你是一個 Python 資料科學家。我有一個環境部空氣品質開放資料的 API 網址：
https://data.moenv.gov.tw/api/v2/aqx_p_432?api_key={API_KEY}&limit=1000&format=JSON
（金鑰已經存在變數 API_KEY 裡）
我不會寫程式，請幫我寫出一段可以在 Google Colab 運行的 Python 程式碼，
功能是要抓取這個網址的資料，並將裡面的監測時間、PM2.5、O3 欄位整理成一個乾淨的表格（DataFrame）顯示出來。
請在程式碼裡加上詳細的中文註解。
```
**改良版（好 Prompt 的 6 個零件：① 任務 ② 資料來源＋樣子 ③ 英文欄位 ④ 輸出 ⑤ 限制 ⑥ 驗證）**：
```
【① 任務】我想知道花蓮站現在的空氣品質，並和全臺其他測站比較 PM2.5、O3、NO2。
【② 資料來源＋樣子】環境部 API：https://data.moenv.gov.tw/api/v2/aqx_p_432?api_key={API_KEY}&limit=1000&format=JSON
（金鑰已用 userdata.get 存在變數 API_KEY）。回傳是 JSON 清單，每筆像這樣：（貼上步驟 1 印出的樣本）
【③ 欄位】sitename、county、publishtime、pm2.5、o3、no2（數值是文字）
【④ 輸出】整理成 DataFrame，數值轉成數字
【⑤ 限制】你是 Python 資料科學家；程式要能在 Google Colab 執行，加上詳細中文註解
【⑥ 驗證】先印出欄位名稱和前 5 筆讓我確認，再印出總共幾個測站
```
> 不用背語法，但要會描述需求、看懂結果、發現錯誤：AI 給的結果，先看前 5 筆對不對再往下。

In [5]:
# ⬇️ 把 AI 生成的程式碼放在這一格，執行看看（▶）

In [6]:
#@title 🔑 參考解答（卡關 5 分鐘以上再展開）：抓現況資料
import requests, pandas as pd
import moenv_api as ma                      # 代碼庫的 get_json 會處理政府網站憑證的小問題
data = ma.get_json(url)                     # 1. 向 API 要資料（回傳 JSON 清單）
df_now = pd.DataFrame(data)                 # 2. 轉成表格
table = df_now[['sitename', 'county', 'publishtime', 'pm2.5', 'o3']]   # 3. 只留需要的欄位
print('全臺測站數：', len(df_now))
table.head(10)

全臺測站數： 84


,sitename,county,publishtime,pm2.5,o3
0,基隆,基隆市,2026/10/18 15:00:00,6,29
1,汐止,新北市,2026/10/18 15:00:00,6,30
2,新店,新北市,2026/10/18 15:00:00,14,75
3,土城,新北市,2026/10/18 15:00:00,19,80
4,板橋,新北市,2026/10/18 15:00:00,17,77
5,新莊,新北市,2026/10/18 15:00:00,19,75
6,菜寮,新北市,2026/10/18 15:00:00,10,54
7,林口,新北市,2026/10/18 15:00:00,24,75
8,淡水,新北市,2026/10/18 15:00:00,21,36
9,士林,臺北市,2026/10/18 15:00:00,10,49


## 步驟 3｜除錯：把錯誤訊息丟回給 AI
程式出錯時，**不要慌**，照這三步：
1. 看最後一行紅字（錯誤類型＋訊息）　2. 按錯誤下方的 **「說明錯誤」**（或複製紅字）　3. 用下面的 Prompt 問 AI
```
我在 Colab 執行你給的程式碼，出現下面的錯誤。請用中文說明原因，並給我修正後的完整程式碼：
（貼上紅字錯誤訊息）
```
| 常見錯誤 | 通常的原因 | 怎麼修 |
|---|---|---|
| `NameError: name 'API_KEY' is not defined` | 沒先執行步驟 0 | 回到步驟 0 執行 |
| `JSONDecodeError` 或回傳「api_key 不存在。」 | 金鑰錯誤或沒放進網址 | 檢查 🔑 密鑰名稱與值 |
| `KeyError: 'PM2.5'` | 欄位名稱大小寫不同（實際是 `pm2.5`） | 先 `print(df.columns)` 再改 |
| `SSLError … Missing Subject Key Identifier` | 政府網站憑證＋新版 Python 嚴格檢查 | 改用 `ma.get_json(url)` |

In [7]:
# 練習：這格故意有錯（欄位名稱大小寫），執行後把錯誤丟給 AI 修正
df_now[['SiteName', 'PM2.5', 'O3']].head()

KeyError: "None of [Index(['SiteName', 'PM2.5', 'O3'], dtype='str')] are in the [columns]"

## 步驟 4｜篩出花蓮站（現況）
```
請把上面的 df_now 只留下 sitename 等於「花蓮」的那一列，
並顯示 publishtime、aqi、pm2.5、o3、no2、nox、wind_speed 這幾個欄位。
```

In [8]:
# ⬇️ AI 生成的程式碼放這裡

In [9]:
#@title 🔑 參考解答：篩出花蓮（現況）
hualien_now = df_now[df_now['sitename'] == '花蓮']
hualien_now[['sitename', 'publishtime', 'aqi', 'status', 'pm2.5', 'o3', 'no2', 'nox', 'wind_speed']]

,sitename,publishtime,aqi,status,pm2.5,o3,no2,nox,wind_speed
60,花蓮,2026/10/18 15:00:00,23,良好,0,30,0,1.6,2.6


## 步驟 5｜抓花蓮近一個月的歷史資料（aqx_p_488）
歷史資料集一次最多給 1000 筆，所以要「**只篩花蓮**」＋「**依時間由新到舊排序**」：
```
請幫我寫 Colab 程式碼，用環境部 API 的 aqx_p_488（空氣品質指標歷史資料），
加上參數 filters=sitename,EQ,花蓮 和 sort=datacreationdate desc，limit=1000，
把花蓮站的資料抓下來，轉成 DataFrame，並告訴我資料的最早與最晚時間、總共幾筆。
```

In [10]:
# ⬇️ AI 生成的程式碼放這裡

In [11]:
#@title 🔑 參考解答：抓花蓮近一個月歷史資料
hist = ma.fetch('aqx_p_488', API_KEY, limit=1000,
                filters='sitename,EQ,花蓮', sort='datacreationdate desc')
hist['datacreationdate'] = pd.to_datetime(hist['datacreationdate'])
hist = hist.sort_values('datacreationdate')
last30 = hist[hist['datacreationdate'] > hist['datacreationdate'].max() - pd.Timedelta(days=30)]
print('抓到', len(hist), '筆；近 30 天', len(last30), '筆')
print('時間範圍：', last30['datacreationdate'].min(), '～', last30['datacreationdate'].max())
last30[['datacreationdate', 'aqi', 'pm2.5', 'o3', 'no2', 'nox', 'windspeed']].tail()

抓到 1000 筆；近 30 天 718 筆
時間範圍： 2026-09-18 16:00:00 ～ 2026-10-18 15:00:00


,datacreationdate,aqi,pm2.5,o3,no2,nox,windspeed
4,2026-10-18 11:00:00,15,1,27,1,2.2,2.4
3,2026-10-18 12:00:00,17,1,29,0,1.6,2.8
2,2026-10-18 13:00:00,19,2,30,1,1.8,2.5
1,2026-10-18 14:00:00,21,1,29,0,1.6,2.4
0,2026-10-18 15:00:00,23,0,30,0,1.6,2.6


## 步驟 6｜存成 CSV 並下載到電腦（下週模組活動二要用！）
```
請把 last30 存成 CSV 檔，檔名用 hualien_近30天_今天日期.csv，編碼用 utf-8-sig（Excel 開才不會亂碼），然後下載到我的電腦。
```

In [12]:
# ⬇️ AI 生成的程式碼放這裡

In [13]:
#@title 🔑 參考解答：匯出 CSV 並下載
from datetime import date
fname = f'hualien_近30天_{date.today():%Y%m%d}.csv'
last30.to_csv(fname, index=False, encoding='utf-8-sig')
print('✅ 已存檔：', fname, '／', len(last30), '列 ×', last30.shape[1], '欄')
try:
    from google.colab import files
    files.download(fname)                 # 瀏覽器會跳出下載
except Exception:
    pass

✅ 已存檔： hualien_近30天_20261018.csv ／ 718 列 × 25 欄


## 📖 欄位的化學意義（課堂講解）
| 欄位 | 中文 | 單位 | 環境化學重點 |
|---|---|---|---|
| `o3` | 臭氧 | ppb | 二次污染物：NO₂ 光解產生 O 原子，O + O₂ → O₃；白天、日照強時高 |
| `no2` / `no` / `nox` | 二氧化氮／一氧化氮／氮氧化物 | ppb | 燃燒（交通）排放；NOx = NO + NO₂；NO 會「滴定」消耗 O₃ |
| `pm2.5` | 細懸浮微粒 | μg/m³ | 一次排放＋二次生成（硫酸鹽、硝酸鹽、有機氣膠） |
| `so2` | 二氧化硫 | ppb | 含硫燃料燃燒；可氧化成硫酸鹽 PM |
| `windspeed` / `winddirec` | 風速／風向 | m/s／度 | 稀釋與傳輸；判斷污染是本地還是外來 |

### ✅ 自我檢核
- [ ] 我的金鑰只放在 🔑 密鑰裡，沒有出現在任何程式碼、截圖或分享連結
- [ ] 我看到了全臺測站表、花蓮現況 1 列、花蓮近 30 天約 700 列
- [ ] CSV 已下載，並記錄了我給 AI 的每一個 Prompt

➡️ 課後學習單：[第 13 週 模組活動一](https://rlirdo.github.io/envchem-ai-module-kit/worksheets/W13_M1.html)

<sub>NDHU 自資系 游豐兆 博士生製作｜指導教授 楊悠娟 副教授｜國立東華大學 115-1 環境化學 × 生成式 AI 融入專業課程計畫</sub>